In [0]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import rand, col, when, broadcast, concat, lit

In [0]:
# Create a new SparkSession
spark = (SparkSession
         .builder
         .appName("avoid-data-skew")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "512m")
         .getOrCreate())

# Set log level to ERROR
spark.sparkContext.setLogLevel("ERROR")

In [0]:
# Define a function to measure the execution time of a query
import time

def measure_time(query):
    start = time.time()
    query.collect() # Force the query execution by calling an action
    end = time.time()
    print(f"Execution time: {end - start} seconds")

In [0]:
# Create some sample data frames
# A large data frame with 10 million rows and two columns: id and value
large_df = spark.range(0, 10000000).withColumn("value", rand(seed=42))

# A skewed data frame with 1 million rows and two columns: id and value
skewed_df = spark.range(0, 1000000).withColumn("value", rand(seed=42)).withColumn("id", when(col("id")%4 == 0, 0).otherwise(col("id")))

In [0]:
large_df_repartitioned = large_df.repartition(5, "id")
num_partitions = large_df_repartitioned.rdd.getNumPartitions()
print(f"Number of partitions: {num_partitions}")

partition_sizes = large_df_repartitioned.rdd.glom().map(len).collect()
print(f"Partition sizes: {partition_sizes}")

skewed_df_repartitioned = skewed_df.repartition(5, "id")
num_partitions = skewed_df_repartitioned.rdd.getNumPartitions()
print(f"Number of partitions: {num_partitions}")

partition_sizes = skewed_df_repartitioned.rdd.glom().map(len).collect()
print(f"Partition sizes: {partition_sizes}")

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [0]:
# Join the non-skewed DataFrames using the default join strategy (sort-merge join)
inner_join_df = large_df_repartitioned.join(skewed_df_repartitioned, "id")
measure_time(inner_join_df)

In [0]:
inner_join_df.explain()

### Isolate skewed data

In [0]:
# Identify the skewed value in the invoice_id column
skewed_value = 0

# Filter out the rows with the skewed value from both DataFrames
large_skewed_df = large_df_repartitioned.filter(large_df_repartitioned.id == skewed_value)
small_skewed_df = skewed_df_repartitioned.filter(skewed_df_repartitioned.id == skewed_value)

# Filter out the rows without the skewed value from both DataFrames
large_non_skewed_df = large_df_repartitioned.filter(large_df_repartitioned.id != skewed_value)
small_non_skewed_df = skewed_df_repartitioned.filter(skewed_df_repartitioned.id != skewed_value)

# Join the non-skewed DataFrames using the default join strategy (sort-merge join)
non_skewed_join_df = large_non_skewed_df.join(small_non_skewed_df, "id")

# Join the skewed DataFrames using a broadcast hash join
skewed_join_df = large_skewed_df.join(broadcast(small_skewed_df), "id")

# Union the results from both joins
final_join_df = non_skewed_join_df.union(skewed_join_df)

measure_time(final_join_df)


In [0]:
final_join_df.explain()

### Broadcast hash join

In [0]:
smaller_df = skewed_df_repartitioned

# Use the broadcast function to mark the smaller DataFrame for broadcasting
from pyspark.sql.functions import broadcast
broadcast_df = broadcast(smaller_df)

# Join the two DataFrames using the broadcast function as an argument

broadcast_join_df = large_df_repartitioned.join(broadcast_df, "id")

measure_time(broadcast_join_df)

In [0]:
broadcast_join_df.explain()

In [0]:
# Display some rows of the result
broadcast_join_df.show(10)

### Key salting

In [0]:
# Import random module
import random

# Identify the skewed value in the id column
skewed_value = 0

# Create a list of salt values to append to the skewed value
salt_list = ["_A", "_B", "_C", "_D", "_E"]

# Create a new column in both DataFrames that contains the original invoice_id value plus a salt value if it is skewed, or just the original invoice_id value otherwise
large_df = (large_df_repartitioned
              .withColumn("salted_id", when(large_df_repartitioned.id == skewed_value, concat(large_df_repartitioned.id, lit(random.choice(salt_list))))
                          .otherwise(large_df_repartitioned.id)))
skewed_df = (skewed_df_repartitioned
             .withColumn("salted_id", when(skewed_df_repartitioned.id == skewed_value, concat(skewed_df_repartitioned.id, lit(random.choice(salt_list))))
                         .otherwise(skewed_df_repartitioned.id)))

# Join the two DataFrames on the new column using the default join strategy (sort-merge join)
salted_join_df = large_df.join(skewed_df, "salted_id")

# Drop the new column and keep only the original invoice_id column
final_join_df = salted_join_df.drop("salted_id")

measure_time(final_join_df)

In [0]:
# Display some rows of the result
final_join_df.show(10)

In [0]:
spark.stop()